# Hyperparameter Tuning

## Objective

The objective of this notebook is to optimize the performance of the best-performing model obtained during model training.

Gradient Boosting Regressor achieved the highest R² score in the previous notebook. In this notebook, GridSearchCV is used to search for the best combination of hyperparameters.

The tuned model will be evaluated and saved for further analysis.

In [4]:
# ==========================
# Import Libraries
# ==========================

import joblib
import pandas as pd

from sklearn.ensemble import GradientBoostingRegressor

from sklearn.model_selection import GridSearchCV

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [5]:
# ==========================
# Load Models
# ==========================

X_train = joblib.load("../models/X_train_encoded.pkl")
X_test = joblib.load("../models/X_test_encoded.pkl")

y_train = joblib.load("../models/y_train.pkl")
y_test = joblib.load("../models/y_test.pkl")

In [6]:
# ==========================
# Training Baseline Model
# ==========================

baseline_model = GradientBoostingRegressor(random_state = 42)

baseline_model.fit(X_train, y_train)

baseline_predictions = baseline_model.predict(X_test)

baseline_r2 = r2_score(y_test, baseline_predictions)

print(f"Baseline R² Score: {baseline_r2:.4f}")

Baseline R² Score: 0.9077


In [7]:
# ==========================
# Defining Hyperparameter Grid
# ==========================

param_grid = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 4, 5],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

In [8]:
# ==========================
# Grid Search
# ==========================

grid_search = GridSearchCV(
    estimator = GradientBoostingRegressor(random_state = 42),
    param_grid = param_grid,
    scoring = 'r2',
    cv = 5,
    n_jobs = 1,
    verbose = 2
)

grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 108 candidates, totalling 540 fits
[CV] END learning_rate=0.01, max_depth=3, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   0.5s
[CV] END learning_rate=0.01, max_depth=3, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   0.3s
[CV] END learning_rate=0.01, max_depth=3, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   0.4s
[CV] END learning_rate=0.01, max_depth=3, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   0.3s
[CV] END learning_rate=0.01, max_depth=3, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   0.3s
[CV] END learning_rate=0.01, max_depth=3, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=   0.7s
[CV] END learning_rate=0.01, max_depth=3, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=   0.7s
[CV] END learning_rate=0.01, max_depth=3, min_samples_leaf=1, min_samples_split=2, n_estimators=2

KeyboardInterrupt: 

In [ ]:
print(f"Best Parameters: {grid_search.best_params_}")
print()
print(f"Best CV Score: {grid_search.best_score_}")

In [ ]:
# ==========================
# Evaluating Tuned Model
# ==========================

best_model = grid_search.best_estimator_

predictions = best_model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
mse = mean_squared_error(y_test, predictions)
rmse = mse ** 0.5
r2 = r2_score(y_test, predictions)

print(f"MAE  : {mae:.2f}")
print(f"MSE  : {mse:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R²   : {r2:.4f}")

In [ ]:
# ==========================
# Comparing Results
# ==========================

comparison = pd.DataFrame({
    'Model': ["Baseline Gradient Boosting", "Tuned Gradient Boosting"],
    'R2 Score': [baseline_r2, r2]
})

comparison